# GDELT Consistent Ticker Scanner (Colab)

Colab version of `discover_gdelt_consistent_tickers.py`.

## What it does
- Reads ticker universe (TXT/CSV/JSON)
- Queries GDELT (via `gdeltdoc`) per ticker
- Computes consistency stats (weekly/monthly)
- Writes keep rows **incrementally while running**
- Saves full summary CSV at the end

Tip for parallel Colab runs: split your universe file into shards and run one shard per notebook.


In [ ]:
!pip -q install gdeltdoc pandas requests pyarrow

In [ ]:
import json
import math
import re
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd
from gdeltdoc import GdeltDoc, Filters
from google.colab import files


In [ ]:
# ===== USER CONFIG =====
START_DATE = "2018-01-01"
END_DATE = "2025-12-31"
FREQ = "M"  # "W" or "M"

MIN_ARTICLES_PER_PERIOD = 4.0
MIN_PASS_RATIO = 0.60
MIN_TOTAL_ARTICLES = 100
MIN_ACTIVE_RATIO = 0.40

MIN_REQUEST_INTERVAL = 7.0
MAX_RETRIES = 3
MAX_BACKOFF_SECONDS = 180.0
SLEEP_SECONDS = 0.0

SHOW_PREVIEW_PERIODS = 6
DEBUG_TICKERS = {"AAPL"}  # set() for none
DEBUG_RAW_LIMIT = 8

# Optional cap for quick test; set 0 to run all in uploaded universe
MAX_TICKERS = 0

OUTPUT_DIR = Path("/content")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Upload your universe file (TXT/CSV/JSON)
# Example TXT line format:
# AAPL\tApple Inc.\tNASDAQ
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded")

UNIVERSE_FILE = list(uploaded.keys())[0]
print("Using universe file:", UNIVERSE_FILE)


In [ ]:
@dataclass
class Thresholds:
    min_articles_per_period: float
    min_pass_ratio: float
    min_total_articles: int
    min_active_ratio: float


def load_universe(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        df = pd.DataFrame(obj)
    elif path.suffix.lower() == ".txt":
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                parts = [p.strip() for p in re.split(r"[\t,]", line) if p.strip()]
                ticker = parts[0]
                company_name = parts[1] if len(parts) >= 2 else ticker
                exchange = parts[2] if len(parts) >= 3 else "UNKNOWN"
                rows.append({"ticker": ticker, "company_name": company_name, "exchange": exchange})
        df = pd.DataFrame(rows)
    else:
        raise ValueError("Universe file must be TXT/CSV/JSON")

    if "ticker" not in df.columns:
        raise ValueError("Universe file requires ticker column")
    if "company_name" not in df.columns:
        df["company_name"] = df["ticker"]
    if "exchange" not in df.columns:
        df["exchange"] = "UNKNOWN"

    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["company_name"] = df["company_name"].astype(str).str.strip()
    return df[["ticker", "company_name", "exchange"]].drop_duplicates(subset=["ticker"]).reset_index(drop=True)


def build_keyword_candidates(company_name: str, ticker: str):
    raw = (company_name or "").strip()
    cleaned = re.sub(r"\s+", " ", raw).strip()
    cleaned = re.sub(
        r"\s*-\s*(common stock|ordinary shares?|american depositary shares?.*|class [a-z].*|units?.*|warrants?.*)$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\(.*?\)", "", cleaned).strip()
    first_part = cleaned.split(" - ")[0].split(",")[0].strip()
    short_words = " ".join(first_part.split()[:4]).strip()
    cands = [cleaned, first_part, short_words, ticker]
    out, seen = [], set()
    for c in cands:
        c = re.sub(r"\s+", " ", str(c or "")).strip()
        if not c or len(c) < 3:
            continue
        if len(c) > 90:
            c = c[:90].rsplit(" ", 1)[0].strip() or c[:90]
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out or [ticker]


def timeline_df_to_daily_series(timeline_df: pd.DataFrame, start_date: str, end_date: str) -> pd.Series:
    idx = pd.date_range(pd.to_datetime(start_date), pd.to_datetime(end_date), freq="D")
    out = pd.Series(0.0, index=idx)
    if timeline_df is None or timeline_df.empty:
        return out

    date_col = "datetime" if "datetime" in timeline_df.columns else timeline_df.columns[0]
    val_col = None
    for c in ["Article Count", "article_count", "value", "count", "Count"]:
        if c in timeline_df.columns:
            val_col = c
            break
    if val_col is None:
        for c in timeline_df.columns:
            lc = c.lower()
            if c == date_col:
                continue
            if "article" in lc and "all articles" not in lc:
                val_col = c
                break
    if val_col is None:
        return out

    tmp = timeline_df[[date_col, val_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
    if pd.api.types.is_datetime64tz_dtype(tmp[date_col]):
        tmp[date_col] = tmp[date_col].dt.tz_convert("UTC").dt.tz_localize(None)
    tmp[val_col] = pd.to_numeric(tmp[val_col], errors="coerce")
    tmp = tmp.dropna(subset=[date_col, val_col])
    if tmp.empty:
        return out
    tmp["d"] = tmp[date_col].dt.normalize()
    daily = tmp.groupby("d")[val_col].sum()
    for d, v in daily.items():
        if d in out.index:
            out.loc[d] = float(v)
    return out


def evaluate_consistency(daily_counts: pd.Series, freq: str, th: Thresholds) -> dict:
    period = daily_counts.resample("W-MON").sum() if freq == "W" else daily_counts.resample("MS").sum()
    periods_total = int(len(period))
    total_articles = float(period.sum())
    periods_active = int((period > 0).sum())
    periods_pass = int((period >= th.min_articles_per_period).sum())

    active_ratio = periods_active / periods_total if periods_total else 0.0
    pass_ratio = periods_pass / periods_total if periods_total else 0.0
    keep = (
        total_articles >= th.min_total_articles
        and pass_ratio >= th.min_pass_ratio
        and active_ratio >= th.min_active_ratio
    )
    return {
        "periods_total": periods_total,
        "total_articles": round(total_articles, 2),
        "periods_active": periods_active,
        "periods_pass": periods_pass,
        "active_ratio": round(active_ratio, 4),
        "pass_ratio": round(pass_ratio, 4),
        "avg_articles_per_period": round(float(period.mean()) if periods_total else 0.0, 3),
        "median_articles_per_period": round(float(period.median()) if periods_total else 0.0, 3),
        "p25_articles_per_period": round(float(period.quantile(0.25)) if periods_total else 0.0, 3),
        "keep": bool(keep),
    }


def append_csv_row(path: Path, row: dict, columns: list[str]):
    exists = path.exists()
    pd.DataFrame([{c: row.get(c, "") for c in columns}], columns=columns).to_csv(
        path, mode="a", index=False, header=not exists
    )


In [ ]:
# Run scanner
universe = load_universe(Path(UNIVERSE_FILE))
if MAX_TICKERS and MAX_TICKERS > 0:
    universe = universe.head(MAX_TICKERS).copy()

print("Universe size:", len(universe))
th = Thresholds(MIN_ARTICLES_PER_PERIOD, MIN_PASS_RATIO, MIN_TOTAL_ARTICLES, MIN_ACTIVE_RATIO)
gd = GdeltDoc()
rate_state = {"last_request_ts": 0.0}

tag = (
    f"{FREQ}_{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}_"
    f"minp{MIN_ARTICLES_PER_PERIOD}_mpr{MIN_PASS_RATIO}_mint{MIN_TOTAL_ARTICLES}"
)
summary_path = OUTPUT_DIR / f"gdelt_ticker_consistency_summary_{tag}.csv"
keep_full_path = OUTPUT_DIR / f"gdelt_ticker_keep_list_{tag}.csv"
keep_tickers_path = OUTPUT_DIR / f"gdelt_ticker_keep_tickers_{tag}.csv"

if keep_full_path.exists():
    keep_full_path.unlink()

rows = []
keep_columns = [
    "ticker", "company_name", "exchange", "periods_total", "total_articles",
    "periods_active", "periods_pass", "active_ratio", "pass_ratio",
    "avg_articles_per_period", "median_articles_per_period", "p25_articles_per_period",
    "avg_articles_per_month", "median_articles_per_month", "keep",
]

for i, rec in universe.iterrows():
    ticker = rec["ticker"]
    company_name = rec["company_name"] if rec["company_name"] else ticker
    print(f"[{i+1}/{len(universe)}] Processing {ticker} ...", flush=True)

    last_exc = None
    daily = pd.Series(dtype=float)
    for kw in build_keyword_candidates(company_name, ticker):
        print(f"[query] {ticker} using keyword: {kw}", flush=True)
        success = False
        for attempt in range(MAX_RETRIES + 1):
            try:
                elapsed = time.time() - rate_state.get("last_request_ts", 0.0)
                if elapsed < MIN_REQUEST_INTERVAL:
                    time.sleep(MIN_REQUEST_INTERVAL - elapsed)

                filters = Filters(keyword=kw, start_date=START_DATE, end_date=END_DATE, language="english")
                timeline_df = gd.timeline_search("timelinevolraw", filters)
                rate_state["last_request_ts"] = time.time()
                if ticker in DEBUG_TICKERS:
                    print(f"[debug] {ticker} cols: {list(timeline_df.columns)}", flush=True)
                    print(timeline_df.head(DEBUG_RAW_LIMIT).to_string(index=False), flush=True)

                daily = timeline_df_to_daily_series(timeline_df, START_DATE, END_DATE)
                if SLEEP_SECONDS > 0:
                    time.sleep(SLEEP_SECONDS)
                success = True
                break
            except Exception as exc:  # noqa: BLE001
                last_exc = exc
                msg = str(exc).lower()
                invalid_query = (
                    "query was not valid" in msg
                    or "too short or too long" in msg
                    or "specified phrase is too short" in msg
                )
                if invalid_query:
                    print(f"[query-fallback] invalid phrase for {ticker}: '{kw}'", flush=True)
                    break
                backoff = min(max(MIN_REQUEST_INTERVAL * (2 ** attempt), 2.0), MAX_BACKOFF_SECONDS)
                print(f"[retry] sleeping {backoff:.1f}s (attempt {attempt+1}/{MAX_RETRIES+1}) | {exc}", flush=True)
                time.sleep(backoff)
                if attempt == MAX_RETRIES:
                    break
        if success:
            break

    if daily.empty and last_exc is not None:
        print(f"[{i+1}/{len(universe)}] {ticker} -> ERROR: {last_exc}", flush=True)
        rows.append({
            "ticker": ticker,
            "company_name": company_name,
            "exchange": rec.get("exchange", "UNKNOWN"),
            "periods_total": math.nan,
            "total_articles": math.nan,
            "periods_active": math.nan,
            "periods_pass": math.nan,
            "active_ratio": math.nan,
            "pass_ratio": math.nan,
            "avg_articles_per_period": math.nan,
            "median_articles_per_period": math.nan,
            "p25_articles_per_period": math.nan,
            "avg_articles_per_month": math.nan,
            "median_articles_per_month": math.nan,
            "keep": False,
            "error": str(last_exc),
        })
        continue

    period = daily.resample("W-MON").sum() if FREQ == "W" else daily.resample("MS").sum()
    monthly = daily.resample("MS").sum()
    stats = evaluate_consistency(daily, FREQ, th)
    row = {
        "ticker": ticker,
        "company_name": company_name,
        "exchange": rec.get("exchange", "UNKNOWN"),
        **stats,
        "avg_articles_per_month": round(float(monthly.mean()) if len(monthly) else 0.0, 3),
        "median_articles_per_month": round(float(monthly.median()) if len(monthly) else 0.0, 3),
    }
    rows.append(row)

    if row["keep"]:
        append_csv_row(keep_full_path, row, keep_columns)

    keep_label = "KEEP" if row["keep"] else "DROP"
    preview = ", ".join([f"{idx.strftime('%Y-%m')}:{int(v)}" for idx, v in period.tail(SHOW_PREVIEW_PERIODS).items()])
    unit = "week" if FREQ == "W" else "month"
    print(
        f"[{i+1}/{len(universe)}] {ticker} -> {keep_label} | "
        f"total={row['total_articles']:.0f} | pass_ratio={row['pass_ratio']:.2f} | "
        f"active_ratio={row['active_ratio']:.2f} | avg/{unit}={row['avg_articles_per_period']:.2f} | "
        f"avg/month={row['avg_articles_per_month']:.2f} | recent={preview}",
        flush=True,
    )

result = pd.DataFrame(rows)
if not result.empty:
    result = result.sort_values(["keep", "pass_ratio", "total_articles"], ascending=[False, False, False])
result.to_csv(summary_path, index=False)
result[result["keep"] == True][["ticker", "company_name", "exchange"]].to_csv(keep_tickers_path, index=False)  # noqa: E712

print("\nDone.")
print("Summary:", summary_path)
print("Keep list (streamed/full):", keep_full_path)
print("Keep tickers (compact):", keep_tickers_path)
print("Kept:", int((result["keep"] == True).sum()), "/", len(result))  # noqa: E712


In [ ]:
# Download outputs
files.download(str(summary_path))
if keep_full_path.exists():
    files.download(str(keep_full_path))
if keep_tickers_path.exists():
    files.download(str(keep_tickers_path))
